# Team Onboarding: aggdisagg

Welcome to the Python team! This notebook helps you transition from R's `tempdisagg` to `aggdisagg`, a modern Polars-first library for temporal disaggregation and aggregation.

It provides **perfect consistency** (low-freq aggregates exactly match high-freq results), supports all classic methods, and adds modern features like hierarchical reconciliation, uncertainty, and ensemble methods.

## Why switch?
- Native Polars (fast, lazy)
- Pandas/xarray compatibility
- scikit-learn style API
- No R dependencies
- Built-in support for indicators, negatives, uncertainty

## 1. Installation

In [ ]:
# !pip install aggdisagg[all]
# or uv pip install "aggdisagg[all]"

## 2. Basic Disaggregation (equivalent to R td())

In R:
```r
library(tempdisagg)
model <- td(sales.a ~ 0 + exports.q, method="chow-lin-maxlog", conversion="sum")
```

In [ ]:
import polars as pl
from datetime import date
from aggdisagg import TemporalAligner

df = pl.DataFrame({
    "date": [date(2020,1,1), date(2021,1,1), date(2022,1,1)],
    "y": [1200.0, 1500.0, 1350.0],  # low-freq target (yearly sales)
    "ind": [100.0, 125.0, 110.0]     # high-freq indicator (quarterly exports, repeated or actual)
})

aligner = TemporalAligner(
    method="chow-lin-opt",  # auto rho like maxlog
    target_freq="1mo",      # monthly
    agg="sum",              # like conversion="sum"
    indicator_cols=["ind"]
)

monthly = aligner.fit_transform(df, datetime_col="date", target_col="y")
print(monthly.head(6))

## 3. Symmetric Aggregation (new in Python)

R tempdisagg is mostly one-way. aggdisagg gives you `aggregate()` for free.

In [ ]:
yearly_back = aligner.aggregate(monthly, freq="1y")
print("Original y:", df["y"].to_list())
print("Re-aggregated:", yearly_back["y_1y"].to_list())
print("Match:", (yearly_back["y_1y"] - df["y"]).abs().sum() < 1e-8)

## 4. Hierarchical Reconciliation

For multi-level data (e.g., national -> state -> county).

In [ ]:
national = pl.DataFrame({"level": ["nat"], "y": [4050.0]})
regional = pl.DataFrame({"level": ["east", "west"], "y": [2000.0, 2050.0]})

reconciled = aligner.reconcile_hierarchical([national, regional], method="proportional")
print(reconciled[-1])

## 5. Uncertainty Quantification

Get bootstrap or analytic standard errors.

In [ ]:
mean, std = aligner.predict_with_uncertainty(n_bootstrap=50)
print("Std dev sample:", float(std.mean()) if std is not None else None)

## 6. Negative Correction + Ensemble

Automatically fix negatives and combine methods.

In [ ]:
aligner_ens = TemporalAligner(
    method="uniform",
    use_ensemble=True,
    correct_negatives=True
)
high = aligner_ens.fit_transform(df)
print("Ensemble used:", aligner_ens.use_ensemble)

## 7. Pandas / xarray Compatibility

Works seamlessly with pandas DatetimeIndex (wide or long).

In [ ]:
import pandas as pd

pdf = pd.DataFrame(
    {"y": [1200., 1500.]},
    index=pd.date_range("2020", periods=2, freq="YS")
)
a = TemporalAligner(method="uniform", target_freq="M")
high_p = a.fit_transform(pdf)
print(type(high_p), high_p.shape if hasattr(high_p, 'shape') else len(high_p))

## 8. sktime Compatibility

Use inside sktime pipelines.

In [ ]:
# from aggdisagg import TemporalAligner
# wrapper = aligner.get_sktime_transformer()
# (requires sktime installed)

## Tips for R users

- `method="chow-lin-opt"` ≈ R `chow-lin-maxlog`
- `agg="sum"` ≈ `conversion="sum"`
- Use `indicator_cols` for the `~ indicator` part
- Always check `aligner.summary()` and round-trip with `aggregate()`
- For daily data, set `target_freq="1d"` (library handles calendar)

## Next Steps

- Run `benchmarks/compare.py`
- Explore `docs/` for full reference
- For hierarchical or high-dim data, see the methods in core.py